<a href="https://colab.research.google.com/github/QuentinMokK/amazon-review-sentiment-analysis/blob/main/25162013_Shu_MO_Assessment_2_32513_31005_Advanced_Data_Analytics_Algorithms%2C_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Amazon Musical Instruments Reviews — Sentiment Classification (Option 2)

## References / Attribution
Dataset: Kaggle Amazon Reviews (Musical Instruments).  
I reviewed public notebooks **only for ideas** (label mapping, TF-IDF), then reimplemented
the pipeline with my own preprocessing, CV protocol, model comparison, and error analysis.

---


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

#
PROJECT_DIR = '/content/drive/MyDrive/qwere'

import os, sys, pathlib
os.chdir(PROJECT_DIR)          #
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('PWD =', pathlib.Path.cwd())
!ls -lah


Mounted at /content/drive/
PWD = /content/drive/MyDrive/qwere
total 424M
-rw------- 1 root root 418M Oct  4 15:15 best_model.pth
-rw------- 1 root root 8.1K Oct  2 14:41 demo.py
drwx------ 2 root root 4.0K Oct  3 00:06 .ipynb_checkpoints
-rw------- 1 root root 5.9K Oct  2 14:41 main.py
-rw------- 1 root root 5.3K Oct  2 14:41 model.py
-rw------- 1 root root 5.9M Oct  2 14:35 Musical_instruments_reviews.csv
drwx------ 2 root root 4.0K Oct  2 14:42 __pycache__
-rw------- 1 root root  14K Oct  4 15:19 rating_confusion_matrix.png
-rw------- 1 root root  14K Oct  4 15:19 sentiment_confusion_matrix.png
-rw------- 1 root root 5.5K Oct  2 14:41 setup.py
-rw------- 1 root root  13K Oct  2 14:41 train_evaluate.py


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers scikit-learn pandas numpy matplotlib tqdm nltk wordcloud

#
import nltk
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import pandas as pd

#
df = pd.read_csv('/content/drive/MyDrive/qwere/Musical_instruments_reviews.csv')


print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)



Shape: (10261, 9)
Columns: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A2IBPI20UZIR0U,1384719342,"cassandra tu ""Yeah, well, that's just like, u...","[0, 0]","Not much to write about here, but it does exac...",5.0,good,1393545600,"02 28, 2014"
1,A14VAT5EAX3D9S,1384719342,Jake,"[13, 14]",The product does exactly as it should and is q...,5.0,Jake,1363392000,"03 16, 2013"
2,A195EZSQDW3E21,1384719342,"Rick Bennette ""Rick Bennette""","[1, 1]",The primary job of this device is to block the...,5.0,It Does The Job Well,1377648000,"08 28, 2013"


In [ ]:
#
import importlib

for m in ['setup', 'model', 'train_evaluate', 'main', 'demo']:
    try:
        importlib.import_module(m)
        print(f'Import OK: {m}')
    except Exception as e:
        print(f'Import FAIL: {m} -> {e}')


Import OK: setup
Import OK: model
Import OK: train_evaluate
Import OK: main
Import FAIL: demo -> No module named 'streamlit'


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers scikit-learn pandas numpy matplotlib tqdm nltk wordcloud
!pip -q install streamlit  #
import nltk; nltk.download('stopwords'); nltk.download('wordnet')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 143.2 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
import pandas as pd
df = pd.read_csv('Musical_instruments_reviews.csv')
print('Shape:', df.shape)
print(df.columns.tolist())
df.head(2)


Shape: (10261, 9)
['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A2IBPI20UZIR0U,1384719342,"cassandra tu ""Yeah, well, that's just like, u...","[0, 0]","Not much to write about here, but it does exac...",5.0,good,1393545600,"02 28, 2014"
1,A14VAT5EAX3D9S,1384719342,Jake,"[13, 14]",The product does exactly as it should and is q...,5.0,Jake,1363392000,"03 16, 2013"


In [ ]:
!python main.py


2025-10-05 02:19:08.641557: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759630748.660805    1506 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759630748.666623    1506 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759630748.681085    1506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759630748.681110    1506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759630748.681114    1506 computation_placer.cc:177] computation placer alr

In [ ]:
# Step 1. load Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2.
%cd /content/drive/MyDrive/your_project_folder

# Step 3.
!pip install torch transformers scikit-learn matplotlib seaborn tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Errno 2] No such file or directory: '/content/drive/MyDrive/your_project_folder'
/content


In [ ]:
!python demo.py

2025-10-05 03:28:45.248 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.249 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.313 
  command:

    streamlit run demo.py [ARGUMENTS]
2025-10-05 03:28:45.313 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.313 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.314 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.314 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-05 03:28:45.314 Thread 'MainThr